# Temporal-Resolvability Map

This notebook is the user-run entrypoint for the locked synthetic three-state experiment. The scripts remain the canonical generators; the notebook exposes every run toggle and writes to a separate `*_manual` output directory so existing results are not silently reused or overwritten.

Nothing executes by default. First run the smoke test, inspect its artifacts, then launch the 20-seed locked grid and strict analysis. This experiment uses known synthetic truth and does **not** establish causal identification.

<!-- reviewer-resume-contract -->
## Execution and resume contract

This notebook is aligned with the reviewer-revision implementation. Expensive work is checkpointed and safe to restart with the same configuration. Do not change methods, seeds, thresholds, or output paths while resuming. Saved outputs remain provisional until the compute-machine run and verification gates complete.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'calcium_transient_rising_flank').is_dir():
            return candidate
        nested = candidate / 'calcium-transient-rising-flank'
        if (nested / 'src' / 'calcium_transient_rising_flank').is_dir():
            return nested
    raise RuntimeError('Run from the repository, package root, or notebooks directory.')


PROJECT_ROOT = find_project_root()
PYTHON = sys.executable
print(f'Project root: {PROJECT_ROOT}')
print(f'Python: {PYTHON}')

In [ ]:
def command_text(command: list[str]) -> str:
    return ' '.join(shlex.quote(str(part)) for part in command)


def run_or_print(command: list[str], *, execute: bool) -> None:
    print(command_text(command))
    if not execute:
        print('Dry run only. Set the corresponding RUN_* toggle to True.')
        return
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
    env.setdefault('MPLCONFIGDIR', '/tmp/rising-flanks-matplotlib')
    env.setdefault('XDG_CACHE_HOME', '/tmp/rising-flanks-font-cache')
    subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)

## Run configuration

The locked run uses delays `1,2,4,8`, downsampling `1,2,4`, deadbands `0,1`, and seeds `1–20`. The runner averages every acquisition phase before treating a seed as the independent unit.

In [ ]:
RUN_SMOKE = False
RUN_LOCKED_GRID = False
RUN_STRICT_ANALYSIS = False

OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'validation_results'
SMOKE_OUTPUT_DIR = OUTPUT_ROOT / 'temporal_resolvability_map_manual_smoke'
LOCKED_OUTPUT_DIR = OUTPUT_ROOT / 'temporal_resolvability_map_manual'

NATIVE_DELAYS = '1,2,4,8'
DOWNSAMPLE_FACTORS = '1,2,4'
DEADBANDS = '0,1'
LOCKED_SEEDS = ','.join(str(seed) for seed in range(1, 21))
N_STEPS = 1280
TOLERANCE = 0.02
MIN_RUN_SAMPLES = 2

In [ ]:
smoke_command = [
    PYTHON,
    'examples/temporal_resolvability_map.py',
    '--resume',
    '--seeds', '1,2',
    '--native-delays', '1,8',
    '--downsample-factors', '1,4',
    '--deadbands', DEADBANDS,
    '--n-steps', str(N_STEPS),
    '--tolerance', str(TOLERANCE),
    '--min-run-samples', str(MIN_RUN_SAMPLES),
    '--output-dir', str(SMOKE_OUTPUT_DIR),
]
run_or_print(smoke_command, execute=RUN_SMOKE)


In [ ]:
locked_command = [
    PYTHON,
    'examples/temporal_resolvability_map.py',
    '--resume',
    '--seeds', LOCKED_SEEDS,
    '--native-delays', NATIVE_DELAYS,
    '--downsample-factors', DOWNSAMPLE_FACTORS,
    '--deadbands', DEADBANDS,
    '--n-steps', str(N_STEPS),
    '--tolerance', str(TOLERANCE),
    '--min-run-samples', str(MIN_RUN_SAMPLES),
    '--output-dir', str(LOCKED_OUTPUT_DIR),
]
run_or_print(locked_command, execute=RUN_LOCKED_GRID)


In [ ]:
analysis_command = [
    PYTHON,
    'examples/analyze_temporal_resolvability_map.py',
    '--input-dir', str(LOCKED_OUTPUT_DIR),
    '--output-dir', str(LOCKED_OUTPUT_DIR / 'analysis-output'),
]
if RUN_STRICT_ANALYSIS and not (LOCKED_OUTPUT_DIR / 'resolvability_rows.csv').is_file():
    raise FileNotFoundError('Run the locked grid before the strict analysis.')
run_or_print(analysis_command, execute=RUN_STRICT_ANALYSIS)

## Inspect your saved decision

This cell reads only the manual output directory. It does not fall back to results generated in another session.

In [ ]:
summary_path = LOCKED_OUTPUT_DIR / 'summary.json'
if summary_path.is_file():
    payload = json.loads(summary_path.read_text())
    print(json.dumps(payload['gates'], indent=2))
    print(f"Raw rows: {sum(1 for _ in (LOCKED_OUTPUT_DIR / 'resolvability_rows.csv').open()) - 1}")
    print(f"Analysis report: {LOCKED_OUTPUT_DIR / 'analysis-output' / 'analysis-report.md'}")
else:
    print(f'No manual locked result yet at {summary_path}')